# Polynomial Regression: Comparing Regression Models

This notebook uses the same generated dataset as before and keeps the focus on one idea:

**different regression models fit data differently.**

You will compare:
- **Linear regression** (straight line)
- **Polynomial regression** (curved line)
- **Different polynomial degrees** from **1 to 10**


## 1. Generate the data

The data below follows a **curved pattern** with some random noise added.

This means a straight line may not fit very well.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

np.random.seed(42)

# Same dataset as the original notebook
X = np.linspace(-3, 3, 120).reshape(-1, 1)
noise = np.random.normal(0, 2, size=X.shape[0])
y = 0.5 * X[:, 0]**3 - 2 * X[:, 0]**2 + 3 * X[:, 0] + noise

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

plt.figure(figsize=(6, 4))
plt.scatter(X, y, alpha=0.7)
plt.title("Generated Dataset")
plt.xlabel("x")
plt.ylabel("y")
plt.show()


## 2. Linear regression

Linear regression fits a **straight line**.

Because this dataset is curved, a straight line will usually **underfit** the data.


In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

y_train_pred_linear = linear_model.predict(X_train)
y_test_pred_linear = linear_model.predict(X_test)

train_r2_linear = r2_score(y_train, y_train_pred_linear)
test_r2_linear = r2_score(y_test, y_test_pred_linear)

print(f"Linear Regression Train R²: {train_r2_linear:.3f}")
print(f"Linear Regression Test R²:  {test_r2_linear:.3f}")

x_plot = np.linspace(-3.5, 3.5, 300).reshape(-1, 1)

plt.figure(figsize=(6, 4))
plt.scatter(X_train, y_train, alpha=0.6, label="Training data")
plt.plot(x_plot, linear_model.predict(x_plot), color="red", label="Linear fit")
plt.title("Linear Regression")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()


## 3. Polynomial regression

Polynomial regression adds powers of \(x\), such as:

- degree 2 → \(x^2\)
- degree 3 → \(x^3\)

This allows the model to fit a **curve** instead of just a straight line.


In [ ]:
def poly_model(degree):
    return Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("model", LinearRegression())
    ])

for degree in [2, 3, 5]:
    model = poly_model(degree)
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    print(f"Degree {degree} -> Train R²: {train_r2:.3f}, Test R²: {test_r2:.3f}")

    plt.figure(figsize=(6, 4))
    plt.scatter(X_train, y_train, alpha=0.6, label="Training data")
    plt.plot(x_plot, model.predict(x_plot), color="red", label=f"Degree {degree}")
    plt.title(f"Polynomial Regression (Degree {degree})")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.legend()
    plt.show()


## 4. Compare degrees 1 to 10

This table is useful because it shows:

- **Train R²** → how well the model fits the training data
- **Test R²** → how well the model works on new data

### Simple idea
- If both scores are low → the model is probably **too simple**
- If train is much higher than test → the model may be **overfitting**
- The best model is often where **test R² is highest**


In [ ]:
results = []

for degree in range(1, 11):
    model = poly_model(degree)
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    results.append([degree, train_r2, test_r2])

results_df = pd.DataFrame(results, columns=["Degree", "Train R²", "Test R²"])
results_df


## 5. Quick summary

Use the table above to decide:

- Which degree seems to fit the data best?
- At what point does the model become more complex than needed?
- Is there a degree where **Train R² keeps increasing** but **Test R² stops improving**?

That is the key sign to look for when discussing **overfitting**.


## 6. Student reflection questions

1. Why does linear regression underfit this dataset?
2. Which polynomial degree appears to be the best choice?
3. Which degree seems too complex?
4. What pattern do you notice when comparing **Train R²** and **Test R²**?
